In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
from scipy.stats import ks_2samp
import xarray as xr

In [2]:
# Define directories
Diri = 'Storm_Tracking/'
Output_Diri = 'Figures/'

In [3]:
# Open file
def Open_File(File):
    DF = pandas.read_csv(File)
    DF = DF.drop("Unnamed: 0", axis=1)
    return (DF)

In [4]:
# Open each file
def Files_Open(Scenario, Diri, Subset):
    Data_DF = Open_File(Diri+Scenario+'_Data_'+Subset+'_Output.csv')
    ET_DF = Open_File(Diri+Scenario+'_ET_'+Subset+'_Output.csv')
    Codes_DF = Open_File(Diri+Scenario+'_Codes_Output.csv')
# Edit time format
    Time_Cols = ["ET Begin Time", "ET Complete Time", "Trop Peak Time", "Peak Time", "Genesis Time", "Final Time"]
    for Col in Time_Cols:
        ET_DF[Col] = pandas.to_datetime(ET_DF[Col], errors="coerce")
    Data_DF["Time(Z)"] = pandas.to_datetime(Data_DF["Time(Z)"], errors="coerce")
    return (Data_DF, ET_DF, Codes_DF)

In [5]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)
Lon_Bins = Create_Bins(-100,20,5)
Lat_Bins = Create_Bins(0,60,5)

In [6]:
# Number of years for each climate scenario
Num_Years = numpy.array([90,93,93])

In [7]:
# Open files
Control_Data, Control_ET, Control_Codes = Files_Open("Control", Diri, "SubsetB")
RCP45_Data, RCP45_ET, RCP45_Codes = Files_Open("RCP45", Diri, "SubsetB")
RCP85_Data, RCP85_ET, RCP85_Codes = Files_Open("RCP85", Diri, "SubsetB")

In [8]:
# Bootstrap test for statistical significance between medians
def Bootstrap_Test(A, B, n=10000, Random_Seed=28):
    Actual_Diff = numpy.median(B) - numpy.median(A)
    Boot_Diffs = numpy.zeros(n)
    rng = numpy.random.default_rng(Random_Seed)
# Bootstrap test
    for i in range(n):
        A_Sample = rng.choice(A, size=len(A), replace=True)
        B_Sample = rng.choice(B, size=len(B), replace=True)
        Boot_Diffs[i] = numpy.median(B_Sample) - numpy.median(A_Sample)
# Get confidence interval
    CI_Low, CI_High = numpy.percentile(Boot_Diffs[i], [2.5, 97.5])
# Approximate two sided P value
    if Actual_Diff >= 0:
        P_Val = 2 * numpy.mean(Boot_Diffs <= 0)
    else:
        P_Val = 2 * numpy.mean(Boot_Diffs >= 0)
    P_Val = numpy.clip(P_Val, 0, 1)
    return (P_Val)

In [9]:
# Create subsets for each storm phase
def Phase_Subsets(Data):
    Subset_0 = Data[Data["SLP(hPa)"] <= 1010].reset_index()
    Subset_Trop = Subset_0[Subset_0["Storm Phase"] == "Tropical"]
    Subset_Trans = Subset_0[(Subset_0["Storm Phase"] == "Transition")]
    Subset_Extra = Subset_0[Subset_0["Storm Phase"] == "Extratropical"]
    return (Subset_0, Subset_Trop, Subset_Trans, Subset_Extra)

In [10]:
# Find statistical significance for storm phase cumulative distributions
def Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, Var):
# Create empty arrays
    Medians = numpy.zeros((3,4))
    KS_P_Vals = numpy.zeros((2,4))
    MW_P_Vals = numpy.zeros((2,4))
    Boot_P_Vals = numpy.zeros((2,4))
    Phases = ["All", "Tropical", "Transitioning", "Extratropical"]
#
# Find statistical significance for each phase
    for i in range(4):
# Create subsets
        Control_Phase = Phase_Subsets(Control_Data)[i]
        RCP45_Phase = Phase_Subsets(RCP45_Data)[i]
        RCP85_Phase = Phase_Subsets(RCP85_Data)[i]
# Compute medians
        Medians[0][i] = numpy.median(Control_Phase[Var])
        Medians[1][i] = numpy.median(RCP45_Phase[Var])
        Medians[2][i] = numpy.median(RCP85_Phase[Var])
# Compute statistical significance with KS Test
        KS_P_Vals[0][i] = ks_2samp(Control_Phase[Var], RCP45_Phase[Var])[1]
        KS_P_Vals[1][i] = ks_2samp(Control_Phase[Var], RCP85_Phase[Var])[1]
# Compute statistical significance with bootstrap test
        Boot_P_Vals[0][i] = Bootstrap_Test(Control_Phase[Var], RCP45_Phase[Var])
        Boot_P_Vals[1][i] = Bootstrap_Test(Control_Phase[Var], RCP85_Phase[Var])
#
# Output as dataframe
    Signif_DF = pandas.DataFrame({"Variable": [Var, Var, Var, Var], "Phase": Phases, \
                "Median (Control)": Medians[0], "Median (RCP4.5)": Medians[1], "Median (RCP8.5)": Medians[2], \
                "KS Test (RCP4.5 - Control)": KS_P_Vals[0], "KS Test (RCP8.5 - Control)": KS_P_Vals[1], \
                "Bootstrap Test (RCP4.5 - Control)": Boot_P_Vals[0], "Bootstrap Test (RCP8.5 - Control)": Boot_P_Vals[1]})
    Signif_DF = Signif_DF.round(3)
    return (Signif_DF)

In [13]:
SLP_Signif = Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "SLP(hPa)")
SLP_Signif

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,SLP(hPa),All,990.380,989.435,988.05,0.000,0.000,0.014,0.000
1,SLP(hPa),Tropical,991.480,990.670,989.20,0.000,0.000,0.069,0.000
2,SLP(hPa),Transitioning,975.095,969.800,971.51,0.000,0.000,0.001,0.002
3,SLP(hPa),Extratropical,993.610,992.740,991.39,0.004,0.003,0.489,0.005


In [14]:
Lat_Signif = Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lat")
Lat_Signif

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lat,All,30.53,32.24,32.66,0.000,0.00,0.000,0.000
1,Lat,Tropical,26.73,28.59,29.17,0.000,0.00,0.000,0.000
2,Lat,Transitioning,40.24,41.23,40.76,0.012,0.02,0.004,0.069
3,Lat,Extratropical,47.75,49.75,49.37,0.000,0.00,0.000,0.000


In [15]:
Lon_Signif = Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lon")
Lon_Signif

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,Lon,All,-54.80,-50.795,-53.850,0.0,0.000,0.000,0.015
1,Lon,Tropical,-57.91,-54.320,-56.455,0.0,0.000,0.000,0.001
2,Lon,Transitioning,-50.30,-47.960,-51.040,0.0,0.529,0.007,0.469
3,Lon,Extratropical,-41.85,-36.240,-37.390,0.0,0.000,0.000,0.000


In [19]:
# Create output dataframe
Signif_DF = pandas.concat([SLP_Signif, Lat_Signif, Lon_Signif])
Signif_DF

,Variable,Phase,Median (Control),Median (RCP4.5),Median (RCP8.5),KS Test (RCP4.5 - Control),KS Test (RCP8.5 - Control),Bootstrap Test (RCP4.5 - Control),Bootstrap Test (RCP8.5 - Control)
0,SLP(hPa),All,990.380,989.435,988.050,0.000,0.000,0.014,0.000
1,SLP(hPa),Tropical,991.480,990.670,989.200,0.000,0.000,0.069,0.000
2,SLP(hPa),Transitioning,975.095,969.800,971.510,0.000,0.000,0.001,0.002
3,SLP(hPa),Extratropical,993.610,992.740,991.390,0.004,0.003,0.489,0.005
0,Lat,All,30.530,32.240,32.660,0.000,0.000,0.000,0.000
1,Lat,Tropical,26.730,28.590,29.170,0.000,0.000,0.000,0.000
2,Lat,Transitioning,40.240,41.230,40.760,0.012,0.020,0.004,0.069
3,Lat,Extratropical,47.750,49.750,49.370,0.000,0.000,0.000,0.000
0,Lon,All,-54.800,-50.795,-53.850,0.000,0.000,0.000,0.015
1,Lon,Tropical,-57.910,-54.320,-56.455,0.000,0.000,0.000,0.001


In [20]:
# Output DF to csv file
def Output_File(DF, File_Name):
    DF.to_csv(Output_Diri+File_Name)

In [21]:
Output_File(Signif_DF, 'Table_A2.csv')